# Strands Hosted Agent Setup

This notebook helps you build, deploy, and test a **Strands-based custom agent** on Microsoft Foundry Hosted Agents.

### What this notebook does

1. **Build & Push** - Builds a Docker container image for your Strands agent and pushes it to Azure Container Registry (ACR)
2. **Deploy** - Creates a hosted agent version in your Foundry project, pulling the image from ACR
3. **Test Direct** - Validates the agent works by calling Foundry's Responses API directly with your Azure credentials
4. **Test via APIM** - Routes requests through Azure API Management to test the production gateway path

### Prerequisites

Before running this notebook:
- Complete the **infrastructure deployment** notebook first (creates Foundry resources, APIM, ACR)
- Have the deployment outputs available:
  - Foundry project endpoint
  - ACR login server
  - APIM suffix
  - APIM subscription key
- Docker Desktop must be running
- You must be logged in to Azure CLI (`az login`)

## 1. Build And Push The Container Image

This builds the image locally with Docker and pushes it to ACR.
Run this after completing the prerequisites above.

In [ ]:
# Set these placeholders
ACR_NAME = "xxxx"
IMAGE_NAME = "strands-agent"
IMAGE_TAG = "1.0.0"
ACR_LOGIN_SERVER = f"{ACR_NAME}.azurecr.io"
IMAGE_URI = f"{ACR_LOGIN_SERVER}/{IMAGE_NAME}:{IMAGE_TAG}"

print(f"ACR_NAME={ACR_NAME}")
print(f"ACR_LOGIN_SERVER={ACR_LOGIN_SERVER}")
print(f"IMAGE_NAME={IMAGE_NAME}")
print(f"IMAGE_TAG={IMAGE_TAG}")
print(f"IMAGE_URI={IMAGE_URI}")

### Fill in your configuration

Replace the placeholder values below with your actual Azure resources. These should match the outputs from the infrastructure deployment notebook:
- `ACR_NAME`: Your Azure Container Registry name (e.g., `acrxhep4wovroziu`)
- `IMAGE_NAME`: Container image name for your Strands agent (usually `strands-agent`)
- `IMAGE_TAG`: Semantic version tag for your image (e.g., `1.0.0`)

The computed values (`ACR_LOGIN_SERVER`, `IMAGE_URI`) will be used in subsequent steps.

In [ ]:
# Ensure Docker is running
!az acr login --name {ACR_NAME}

In [ ]:

# Use Docker directly from the notebook
print(f"Running: docker build -t {IMAGE_URI} .")
!docker build -t {IMAGE_URI} --platform linux/amd64 .

print(f"Running: docker push {IMAGE_URI}")
!docker push {IMAGE_URI}

print("Docker build and push completed successfully.")

### Build and push your container

This builds the Strands agent Docker image and pushes it to your ACR. The `--platform linux/amd64` flag ensures the image is compatible with Foundry's Linux-based container hosting.

After pushing, your image will be available at `{ACR_LOGIN_SERVER}/strands-agent:1.0.0` for Foundry to pull and deploy.

## 2. Create Hosted Agent Version

This section creates your Strands agent as a **Foundry Hosted Agent** using the Responses protocol v1.0.0.

### Important notes

- **Agent name**: Must match `hosted_agent_id` from the infrastructure deployment (without the version suffix). 
  - E.g., if you set `hosted_agent_id = 'strands-agent:1'` in main deployment, create the agent with name `strands-agent` here.
  - Foundry auto-assigns the version suffix (`:1`, `:2`, etc.)

- **Model endpoint**: Uses APIM **inference** API for model calls (`https://apim-{suffix}.azure-api.net/inference/models`), not the hosted-agent API.

- **Dependencies**: Install `azure-ai-projects` and `azure-identity` to interact with Foundry APIs.

After creation, your agent will transition through status changes. Once it reaches "Running", it's ready to accept requests in sections 3 and 4.

In [ ]:
pip install azure-ai-projects==2.3.0 --force-reinstall --no-cache-dir

### Create the hosted agent in Foundry

This cell deploys your Strands container as a **Hosted Agent** in your Foundry project. 

**Key configuration:**
- **Protocol**: Responses v1.0.0 (standard for Foundry hosted agents)
- **Resources**: 1 CPU, 2Gi memory (adjust based on your agent needs)
- **Environment variables**: 
  - `AZURE_OPENAI_ENDPOINT`: Points to APIM inference API for your agent to call models
  - `APIM_SUBSCRIPTION_KEY`: APIM subscription key for authentication
  - `AZURE_OPENAI_DEPLOYMENT`: Model name to use (e.g., `gpt-5-mini`)

The agent will be created with a version suffix (auto-assigned by Foundry). After creation, it transitions to "Running" state and is ready to accept requests via the Responses API.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentEndpointProtocol,
    ContainerConfiguration,
 )
from azure.identity import DefaultAzureCredential, AzureCliCredential

# Placeholder values - update before running
PROJECT_ENDPOINT = "https://foundry-agents-xxxxxxx.services.ai.azure.com/api/projects/default-foundry-agents"
AGENT_NAME = "strands-agent"

# APIM inference endpoint for model calls.
AZURE_OPENAI_ENDPOINT = "https://apim-xxxxxxxxxx.azure-api.net/inference/models"
AZURE_OPENAI_API_VERSION = "2024-05-01-preview"
AZURE_OPENAI_DEPLOYMENT = "gpt-5-mini"
APIM_SUBSCRIPTION_KEY = "xxxxxxxxx"

print(f"PROJECT_ENDPOINT: {PROJECT_ENDPOINT}")
print(f"AGENT_NAME: {AGENT_NAME}")

credential = AzureCliCredential()

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
    allow_preview=True,
)

agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=HostedAgentDefinition(
        protocol_versions=[
            ProtocolVersionRecord(protocol=AgentEndpointProtocol.RESPONSES, version="1.0.0")
        ],
        cpu="1",
        memory="2Gi",
        container_configuration=ContainerConfiguration(image=IMAGE_URI),
        environment_variables={
            "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
            "AZURE_OPENAI_API_VERSION": AZURE_OPENAI_API_VERSION,
            "AZURE_OPENAI_DEPLOYMENT": AZURE_OPENAI_DEPLOYMENT,
            "APIM_SUBSCRIPTION_KEY": APIM_SUBSCRIPTION_KEY,
            "LOG_LEVEL": "INFO",
            "STRANDS_LOG_LEVEL": "INFO",
        },
    ),
 )

print(f"Agent created: {agent.name}, version: {agent.version}, id {agent.id}")

## 3. Test The Hosted Agent (Direct)

**Purpose:** Validate your agent works by calling the Foundry Responses API directly using your Azure CLI credential.

This is your baseline test—if this fails, APIM proxy tests will also fail. Use this to confirm:
- Agent reached "Running" status ✅
- Network connectivity to Foundry is working ✅
- Authentication with your credential works ✅
- Agent payload format is correct ✅

**How it works:**
- Endpoint: `{PROJECT_ENDPOINT}/agents/{AGENT_NAME}/endpoint/protocols/openai/responses?api-version=v1`
- Auth: Bearer token with `https://ai.azure.com/.default` audience (from your Azure CLI login)
- Headers: Includes `Foundry-Features: HostedAgents=V1Preview` (required for preview access)

If this test succeeds, you know the agent itself is working correctly.

In [ ]:
import json
import requests
from azure.core.credentials import AccessToken

# Correct endpoint for Responses protocol (per Microsoft Learn)
API_VERSION = "v1"

# Build the correct URL: use agent name only (not name:version)
url = f"{PROJECT_ENDPOINT}/agents/{AGENT_NAME}/endpoint/protocols/openai/responses"

# Get a token for Foundry Responses API
token: AccessToken = credential.get_token("https://ai.azure.com/.default")

headers = {
    "Authorization": f"Bearer {token.token}",
    "Content-Type": "application/json",
    "Foundry-Features": "HostedAgents=V1Preview",  # Required for preview
}

payload = {
    "input": "Hello! What can you help me with? What's the weather in Paris today?",
    "stream": False,
}

print(f"POST {url}?api-version={API_VERSION}")
print(f"Agent Name: {AGENT_NAME}")
print(f"Payload: {json.dumps(payload, indent=2)}")
print()

response = requests.post(url, headers=headers, json=payload, params={"api-version": API_VERSION})
print(f"Status: {response.status_code}")
print(f"Headers: {dict(response.headers)}")
print(f"Response text (first 500 chars): {response.text[:500]}")
print()

if response.status_code == 200:
    try:
        result = response.json()
        print(f"Response JSON:")
        print(json.dumps(result, indent=2))
        if "output_text" in result:
            print(f"\nAgent output:\n{result['output_text']}")
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}")
        print(f"Raw response: {response.text}")
else:
    print(f"Error: {response.status_code}")
    if response.text:
        try:
            print(json.dumps(response.json(), indent=2))
        except:
            print(response.text)

## 4. Test The Hosted Agent Via APIM

**Purpose:** Route your request through Azure API Management instead of calling Foundry directly.

This validates the production-like path:
- Client → APIM Gateway → Foundry Hosted Agent Responses API

**What APIM does automatically:**
- Injects managed identity bearer token for Foundry authentication
- Enforces `Content-Type: application/json` header
- Enforces `api-version: 2025-05-15-preview` query parameter
- Injects `Foundry-Features: HostedAgents=V1Preview` header for preview access
- Your client only needs the APIM subscription key (`api-key` header)

**Configuration details** are in [hosted-agent-policy.xml](../../hosted-agent-policy.xml)—update there if you need to change defaults.

Compare this test result with Section 3 to ensure both paths (direct and APIM-proxied) produce consistent responses.

In [ ]:
import json
import requests

# APIM configuration with defaults
DEFAULT_API_VERSION = "2025-05-15-preview"
DEFAULT_CONTENT_TYPE = "application/json"

# APIM endpoint and version
apim_url = f"https://apim-{APIM_SUFFIX}.azure-api.net/hosted-agent-responses/endpoint/protocols/openai/responses"
api_version = DEFAULT_API_VERSION

# Build the correct URL with api-version query parameter
url = f"{apim_url}?api-version={api_version}"

# Headers for APIM: use api-key header with subscription key
headers = {
    "api-key": APIM_SUBSCRIPTION_KEY,
    "Content-Type": DEFAULT_CONTENT_TYPE,
    "Foundry-Features": "HostedAgents=V1Preview",  # Required for preview
}

payload = {
    "input": "Hello! What can you help me with? What's the weather in Paris today?",
    "stream": False,
}

print(f"POST {url}")
print(f"APIM Suffix: {APIM_SUFFIX}")
print(f"API Version: {api_version}")
print(f"Content-Type: {DEFAULT_CONTENT_TYPE}")
print(f"Headers: api-key={APIM_SUBSCRIPTION_KEY[:20]}..., Content-Type, Foundry-Features")
print(f"Payload: {json.dumps(payload, indent=2)}")
print()

response = requests.post(url, headers=headers, json=payload)
print(f"Status: {response.status_code}")
print(f"Headers: {dict(response.headers)}")
print(f"Response text (first 500 chars): {response.text[:500]}")
print()

if response.status_code == 200:
    try:
        result = response.json()
        print(f"Response JSON:")
        print(json.dumps(result, indent=2))
        if "output_text" in result:
            print(f"\nAgent output:\n{result['output_text']}")
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}")
        print(f"Raw response: {response.text}")
else:
    print(f"Error: {response.status_code}")
    if response.text:
        try:
            print(json.dumps(response.json(), indent=2))
        except:
            print(response.text)